# ClimaCity Paris -- Jour 2
## Spark SQL, Delta Lake et Structured Streaming

**Module** : Traitement de données massives avec Apache Spark et PySpark  
**Durée** : 1 journée (6 heures effectives)  
**Prérequis** : Avoir complété le Jour 1 -- la table `disponibilite_consolidee.parquet` doit être présente dans `data/output/`

---

Ce notebook couvre l'intégralité du Jour 2 du projet ClimaCity Paris.  
Il se divise en deux grandes parties :

- **Partie 1 -- Matin (3 h)** : Spark SQL et l'API de fenêtrage analytique, puis Delta Lake
  pour la persistance transactionnelle (écriture, time-travel, `MERGE INTO`).
- **Partie 2 -- Après-midi (3 h)** : Structured Streaming -- connexion à un flux simulé
  de mises à jour de stations, agrégations sur fenêtres glissantes, gestion des données
  tardives (late data) et déclenchement d'alertes.

> **Convention** : les cellules `# [EXERCICE]` contiennent une consigne à compléter.  
> Les cellules `# [CORRECTION]` proposent une solution -- ne les regardez qu'après avoir tenté.


---
## Section 0 -- Configuration

Même structure de chemins qu'au Jour 1. La table consolidée produite hier est le
point de départ de toutes les analyses.


In [1]:
from pathlib import Path
import time

# ── Chemins ─────────────────────────────────────────────────────────────────
# NB : le notebook supposait un sous-dossier (Path("../data")), mais le
# projet est en realite a plat -- tous les notebooks et data/ sont au meme
# niveau (voir Sessions 1 et 2).
DATA_DIR           = Path("data")
OUTPUT_DIR         = DATA_DIR / "output"
VELIB_CONSOLIDE    = OUTPUT_DIR / "disponibilite_consolidee.parquet"
DELTA_DISPONIBLE   = OUTPUT_DIR / "delta" / "disponibilite"
DELTA_ALERTES      = OUTPUT_DIR / "delta" / "alertes"
STREAM_SOURCE_DIR  = OUTPUT_DIR / "stream_input"    # repertoire surveille par Spark
STREAM_CHECKPOINT  = OUTPUT_DIR / "checkpoints"

for p in [VELIB_CONSOLIDE]:
    assert p.exists(), f"Fichier manquant : {p} -- relancez le Jour 1 (Sessions 1 et 2)"

for p in [DELTA_DISPONIBLE, DELTA_ALERTES, STREAM_SOURCE_DIR, STREAM_CHECKPOINT]:
    p.mkdir(parents=True, exist_ok=True)

# ── Parametres ───────────────────────────────────────────────────────────────
APP_NAME      = "ClimaCity-Paris-Jour2"
SHUFFLE_PARTS = 8
SEED          = 42


In [ ]:
import os, subprocess
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from delta import configure_spark_with_delta_pip

# Meme configuration Java que les jours precedents
java_home = subprocess.check_output(["brew", "--prefix", "openjdk@17"]).decode().strip()
os.environ["JAVA_HOME"] = java_home
os.environ["PATH"] = f"{java_home}/bin:" + os.environ.get("PATH", "")

# Delta Lake requiert le package pip "delta-spark" (meme version que pyspark,
# ex. delta-spark==4.2.0 pour pyspark 4.2.0) :
#   pip install delta-spark==4.2.0
# configure_spark_with_delta_pip() derive automatiquement la bonne coordonnee
# Maven du jar Delta a partir de la version installee -- telechargee une
# fois via internet au premier lancement (~/.ivy2 ensuite en cache local).
builder = (
    SparkSession.builder
    .appName(APP_NAME)
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", SHUFFLE_PARTS)
    .config("spark.driver.memory", "4g")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.ui.showConsoleProgress", "false")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("WARN")

print(f"Spark {spark.version} -- Delta Lake actif")
print(f"Spark UI : http://localhost:4040")


In [3]:
# Chargement de la table consolidée produite au Jour 1
df = spark.read.parquet(str(VELIB_CONSOLIDE))
df.cache()
df.count()   # force la mise en cache

print(f"Table consolidée : {df.count():,} lignes  |  {len(df.columns)} colonnes")
df.printSchema()


Table consolidée : 5,267,322 lignes  |  18 colonnes
root
 |-- nom_station: string (nullable = true)
 |-- code_arr: integer (nullable = true)
 |-- capacite: integer (nullable = true)
 |-- horodatage: timestamp (nullable = true)
 |-- velos_meca: integer (nullable = true)
 |-- velos_elec: integer (nullable = true)
 |-- taux_occupation: double (nullable = true)
 |-- statut: string (nullable = true)
 |-- jour_sem: integer (nullable = true)
 |-- heure: integer (nullable = true)
 |-- est_weekend: boolean (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- humidite_pct: double (nullable = true)
 |-- vent_kmh: double (nullable = true)
 |-- precipitation_mm: double (nullable = true)
 |-- est_pluie: boolean (nullable = true)
 |-- annee: integer (nullable = true)
 |-- mois: integer (nullable = true)



---
# PARTIE 1 -- Spark SQL (matin)

## 1.1 Vues temporaires et premières requêtes SQL

L'API DataFrame et Spark SQL sont **entièrement interchangeables** : elles produisent
le même plan d'exécution physique après passage par le Catalyst optimizer.
Le choix entre les deux est une question de lisibilité et d'habitude.

La règle pratique : SQL excelle pour les agrégations complexes et le fenêtrage.
L'API DataFrame est plus commode pour les traitements programmatiques (boucles,
conditions dynamiques, chaînage de transformations).


In [4]:
# Enregistrement des vues temporaires
# Une vue temporaire n'existe que pour la durée de la session Spark.
# Elle ne copie pas les données -- c'est un alias sur le DataFrame.
df.createOrReplaceTempView("disponibilite")

# Vérification
spark.sql("SHOW VIEWS").show()


+---------+-------------+-----------+
|namespace|     viewName|isTemporary|
+---------+-------------+-----------+
|         |disponibilite|       true|
+---------+-------------+-----------+



In [5]:
# Premiere requete : distribution des statuts par arrondissement :
# Nom de l'arrondissement (code_arr), taux d'occupation moyen, ecart-type
spark.sql("""
    SELECT
        code_arr,
        COUNT(*) AS nb_snapshots,
        ROUND(AVG(taux_occupation), 4) AS taux_occupation_moyen,
        ROUND(STDDEV(taux_occupation), 4) AS taux_occupation_ecart_type
    FROM disponibilite
    GROUP BY code_arr
    ORDER BY code_arr
""").show(40)


+--------+------------+---------------------+--------------------------+
|code_arr|nb_snapshots|taux_occupation_moyen|taux_occupation_ecart_type|
+--------+------------+---------------------+--------------------------+
|    NULL|      376119|               0.3401|                    0.2668|
|       6|       49309|               0.4251|                    0.2781|
|       7|       26551|               0.3924|                    0.2875|
|      29|        3793|               0.5653|                    0.2413|
|      30|        7586|               0.3568|                    0.2509|
|      36|       22758|               0.4512|                    0.2747|
|      37|       15172|               0.4332|                    0.2337|
|      38|       22758|               0.2466|                      0.24|
|      52|       18965|               0.1676|                    0.1712|
|      63|        7586|               0.2868|                    0.2014|
|      77|        3793|                0.319|      

In [6]:
# Les fonctions temporelles SQL sont disponibles directement
# Taux moyen d'occupation, nombre de snapshots, nombre de stations par heure
spark.sql("""
    SELECT
        heure,
        ROUND(AVG(taux_occupation), 4) AS taux_occupation_moyen,
        COUNT(*) AS nb_snapshots,
        COUNT(DISTINCT nom_station) AS nb_stations
    FROM disponibilite
    GROUP BY heure
    ORDER BY heure
""").show(24)


+-----+---------------------+------------+-----------+
|heure|taux_occupation_moyen|nb_snapshots|nb_stations|
+-----+---------------------+------------+-----------+
|    0|               0.3963|       90271|       1385|
|    1|               0.3975|      104141|       1385|
|    2|               0.3985|      202744|       1385|
|    3|               0.3999|      220798|       1385|
|    4|               0.3999|      198582|       1385|
|    5|                0.401|      252742|       1385|
|    6|               0.3924|      205530|       1385|
|    7|               0.3728|      251353|       1385|
|    8|               0.3722|      220806|       1385|
|    9|               0.3747|      223576|       1385|
|   10|               0.3678|      241633|       1385|
|   11|               0.3516|      279137|       1385|
|   12|               0.3496|      198586|       1385|
|   13|               0.3534|      237463|       1385|
|   14|               0.3534|      287468|       1385|
|   15|   

---
## 1.2 Questions métier -- Requêtes analytiques

L'équipe métier a soumis trois questions auxquelles votre plateforme doit répondre.
Nous allons les traiter une par une avec Spark SQL.


### Question 1 : Ruptures en heure de pointe matinale

> Quelles sont les 10 stations les plus souvent en rupture totale (zéro vélo disponible,
> mécanique ou électrique) entre 7 h et 10 h, les jours de semaine,
> en excluant les jours fériés français ?

Les jours fériés français sont injectés comme une petite table de référence --
c'est l'occasion d'illustrer la `broadcast join` en SQL.


In [7]:
# Table de jours feries -- adaptee a la vraie periode des donnees
# (26 nov. 2020 -> 9 fev. 2021, et non 2022-2023 comme suppose a l'origine).
# Seuls deux jours feries francais tombent dans cette fenetre.
# Source : legifrance.gouv.fr
jours_feries = spark.createDataFrame([
    ("2020-12-25",),  # Noel
    ("2021-01-01",),  # Jour de l'An
], ["date_ferie"])

jours_feries = jours_feries.withColumn(
    "date_ferie", F.to_date("date_ferie", "yyyy-MM-dd")
)
jours_feries.createOrReplaceTempView("jours_feries")

print(f"{jours_feries.count()} jours feries enregistres (periode reelle des donnees)")


2 jours feries enregistres (periode reelle des donnees)


In [8]:
# Identifier les 10 stations Velib' les plus en rupture de stock pendant les heures de pointe matinales, en excluant les jours feries.
# On ne s'interesse qu'aux stations tres frequentees, ayant plus de 100 observations (snapshots)
# -> Nom de la station (pas d'identifiant numerique dans nos donnees -- voir Session 1),
#    arrondissement, nombre d'observations (snapshots)
# "Rupture totale" = zero velo disponible, ni mecanique ni electrique.
# "Heures de pointe matinale" = 7h, 8h, 9h (avant 10h).
df_q1 = spark.sql("""
    SELECT
        nom_station,
        code_arr,
        COUNT(*) AS nb_observations,
        SUM(CASE WHEN velos_meca = 0 AND velos_elec = 0 THEN 1 ELSE 0 END) AS nb_ruptures,
        ROUND(100.0 * SUM(CASE WHEN velos_meca = 0 AND velos_elec = 0 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_rupture
    FROM disponibilite
    WHERE heure BETWEEN 7 AND 9
      AND jour_sem BETWEEN 2 AND 6
      AND CAST(horodatage AS DATE) NOT IN (SELECT date_ferie FROM jours_feries)
    GROUP BY nom_station, code_arr
    HAVING COUNT(*) > 100
    ORDER BY pct_rupture DESC, nb_ruptures DESC
    LIMIT 10
""")

df_q1.show(truncate=False)


+-------------------------------+--------+---------------+-----------+-----------+
|nom_station                    |code_arr|nb_observations|nb_ruptures|pct_rupture|
+-------------------------------+--------+---------------+-----------+-----------+
|Manufacture Nationale          |NULL    |345            |345        |100.0      |
|8 Mai 1945 - 10 Juillet 1940   |1057387 |345            |345        |100.0      |
|Porte de Pantin - Petits Ponts |NULL    |345            |345        |100.0      |
|Macdonald - Césaria Evora      |213699  |345            |345        |100.0      |
|Mairie du 20ème                |54000   |345            |345        |100.0      |
|Edouard Vaillant - Galliéni    |213936  |345            |345        |100.0      |
|Gare RER les Ardoines          |NULL    |345            |345        |100.0      |
|Flandrin - Longchamp           |38      |345            |345        |100.0      |
|Vanne - Général de Gaulle      |653131  |345            |345        |100.0      |
|Qua

### Question 2 : Impact de la pluie sur le taux d'occupation

> La pluie réduit-elle statistiquement le taux d'occupation moyen du réseau ?
> De combien de points en moyenne ? L'effet est-il homogène selon les arrondissements ?


In [9]:
# Distribution statistique par quartiles du taux d'occupation en fonction de la meteo
df_q2 = spark.sql("""
    SELECT
        est_pluie,
        COUNT(*) AS nb_snapshots,
        ROUND(AVG(taux_occupation), 4) AS taux_moyen,
        ROUND(PERCENTILE_APPROX(taux_occupation, 0.25), 4) AS q1,
        ROUND(PERCENTILE_APPROX(taux_occupation, 0.5), 4) AS mediane,
        ROUND(PERCENTILE_APPROX(taux_occupation, 0.75), 4) AS q3
    FROM disponibilite
    GROUP BY est_pluie
""")
df_q2.show()


+---------+------------+----------+------+-------+------+
|est_pluie|nb_snapshots|taux_moyen|    q1|mediane|    q3|
+---------+------------+----------+------+-------+------+
|     NULL|     5267322|     0.373|0.1429| 0.3103|0.5714|
+---------+------------+----------+------+-------+------+



In [10]:
# Effet de la pluie par arrondissement
df_q2_arr = spark.sql("""
    SELECT
        code_arr,
        ROUND(AVG(CASE WHEN est_pluie THEN taux_occupation END), 4) AS taux_avec_pluie,
        ROUND(AVG(CASE WHEN NOT est_pluie THEN taux_occupation END), 4) AS taux_sans_pluie,
        ROUND(
            AVG(CASE WHEN est_pluie THEN taux_occupation END) -
            AVG(CASE WHEN NOT est_pluie THEN taux_occupation END), 4
        ) AS delta
    FROM disponibilite
    WHERE code_arr IS NOT NULL
    GROUP BY code_arr
    ORDER BY code_arr
""")
df_q2_arr.show(30)
# Un delta negatif signifie que le taux d'occupation baisse sous la pluie
# (moins de velos empruntes -> plus de velos disponibles -> moins de bornettes libres)


+--------+---------------+---------------+-----+
|code_arr|taux_avec_pluie|taux_sans_pluie|delta|
+--------+---------------+---------------+-----+
|       6|           NULL|           NULL| NULL|
|       7|           NULL|           NULL| NULL|
|      29|           NULL|           NULL| NULL|
|      30|           NULL|           NULL| NULL|
|      36|           NULL|           NULL| NULL|
|      37|           NULL|           NULL| NULL|
|      38|           NULL|           NULL| NULL|
|      52|           NULL|           NULL| NULL|
|      63|           NULL|           NULL| NULL|
|      77|           NULL|           NULL| NULL|
|   10968|           NULL|           NULL| NULL|
|   11375|           NULL|           NULL| NULL|
|   11580|           NULL|           NULL| NULL|
|   13105|           NULL|           NULL| NULL|
|   13191|           NULL|           NULL| NULL|
|   13242|           NULL|           NULL| NULL|
|   13248|           NULL|           NULL| NULL|
|   13283|          

### Question 3 : Saisonnalité intra-journalière

> Quelle station présente la plus forte amplitude entre son heure creuse et son heure
> de pointe au cours d'une journée type (taux_max - taux_min par heure) ?

C'est un cas d'usage typique des **fonctions de fenêtrage**.


In [11]:
# Quelles sont les 15 stations dont le comportement est le plus pendulaire -- presque vides a certaines heures, saturees a d'autres ?
df_q3 = spark.sql("""
    WITH taux_par_heure AS (
        SELECT nom_station, heure, AVG(taux_occupation) AS taux_moyen_heure
        FROM disponibilite
        GROUP BY nom_station, heure
    )
    SELECT
        nom_station,
        ROUND(MAX(taux_moyen_heure) - MIN(taux_moyen_heure), 4) AS amplitude,
        ROUND(MIN(taux_moyen_heure), 4) AS taux_min,
        ROUND(MAX(taux_moyen_heure), 4) AS taux_max
    FROM taux_par_heure
    GROUP BY nom_station
    ORDER BY amplitude DESC
    LIMIT 15
""")
df_q3.show(truncate=False)


+-----------------------------------+---------+--------+--------+
|nom_station                        |amplitude|taux_min|taux_max|
+-----------------------------------+---------+--------+--------+
|Gare du Nord - Denain              |0.7561   |0.114   |0.8701  |
|Gare du Nord - Faubourg Saint-Denis|0.663    |0.1151  |0.7781  |
|Léon - Doudeauville                |0.6065   |0.1534  |0.76    |
|Place de la Madeleine - Royale     |0.5944   |0.2862  |0.8805  |
|Riquet - Marx Dormoy               |0.5742   |0.171   |0.7453  |
|Godot de Mauroy - Madeleine        |0.5647   |0.3099  |0.8746  |
|Danielle Casanova - Place Vendôme  |0.5544   |0.2255  |0.7799  |
|Erasme - Ulm                       |0.5368   |0.0928  |0.6297  |
|Cossonnerie - Sébastopol           |0.5365   |0.2544  |0.7909  |
|Poulet - Barbès                    |0.5265   |0.0998  |0.6262  |
|La Jarry - Place Diderot           |0.5181   |0.0849  |0.603   |
|Porte de Saint-Ouen - Henri Huchard|0.5164   |0.2577  |0.7741  |
|Saint-Jac

In [12]:
# [EXERCICE]
# En utilisant Spark SQL, calculez pour chaque station :
# - le taux d'occupation moyen un jour de semaine sec (est_pluie = false)
# - le taux d'occupation moyen un week-end pluvieux (est_pluie = true)
# - le ratio entre les deux
# Affichez les 10 stations avec le ratio le plus eleve (plus forte difference).
#
# Rappel : est_weekend est un booleen, est_pluie aussi.
# ──────────────────────────────────────────────────────────────────────────

spark.sql("""
    WITH agrege AS (
        SELECT
            nom_station,
            AVG(CASE WHEN NOT est_weekend AND est_pluie = false THEN taux_occupation END) AS taux_semaine_sec,
            AVG(CASE WHEN est_weekend AND est_pluie = true THEN taux_occupation END) AS taux_weekend_pluvieux
        FROM disponibilite
        GROUP BY nom_station
    )
    SELECT
        nom_station,
        ROUND(taux_semaine_sec, 4) AS taux_semaine_sec,
        ROUND(taux_weekend_pluvieux, 4) AS taux_weekend_pluvieux,
        ROUND(taux_weekend_pluvieux / taux_semaine_sec, 4) AS ratio
    FROM agrege
    WHERE taux_semaine_sec IS NOT NULL AND taux_weekend_pluvieux IS NOT NULL
    ORDER BY ratio DESC
    LIMIT 10
""").show()
# NB : tant que la periode meteo ne recouvre pas la periode Velib' (voir
# Session 2), "est_pluie" est nul partout -- ce resultat sera vide pour
# l'instant, et se remplira une fois la meteo re-telechargee sur la bonne
# periode.


+-----------+----------------+---------------------+-----+
|nom_station|taux_semaine_sec|taux_weekend_pluvieux|ratio|
+-----------+----------------+---------------------+-----+
+-----------+----------------+---------------------+-----+



---
## 1.3 Fonctions de fenêtrage analytique

Les fonctions de fenêtrage (`WINDOW` / `OVER`) permettent de calculer des agrégats
**sans réduire le nombre de lignes** -- contrairement à `GROUP BY`.
Elles sont indispensables pour les analyses de séries temporelles.

### Les trois familles de fonctions fenêtrées

```
Ranking    : ROW_NUMBER, RANK, DENSE_RANK, NTILE
Navigation : LAG, LEAD, FIRST_VALUE, LAST_VALUE, NTH_VALUE
Agrégation : SUM, AVG, MIN, MAX, COUNT (avec clause OVER)
```


In [13]:
# Cas concret : pour chaque station, calculer le taux d'occupation
# de la fenetre precedente (LAG) et suivante (LEAD),
# ainsi qu'une moyenne mobile sur 3 snapshots.
# (pas d'identifiant numerique de station dans nos donnees -- on partitionne
# par nom_station, voir Session 1)

fenetre_station = Window.partitionBy("nom_station").orderBy("horodatage")

df_avec_lag = (
    df
    .withColumn("taux_precedent", F.lag("taux_occupation").over(fenetre_station))
    .withColumn("taux_suivant", F.lead("taux_occupation").over(fenetre_station))
    .withColumn(
        "taux_moyenne_mobile_3",
        F.avg("taux_occupation").over(fenetre_station.rowsBetween(-1, 1))
    )
    .select("nom_station", "horodatage", "taux_occupation", "taux_precedent", "taux_suivant", "taux_moyenne_mobile_3")
)
df_avec_lag.show(truncate=False)


+-----------------------+-------------------+---------------+--------------+------------+---------------------+
|nom_station            |horodatage         |taux_occupation|taux_precedent|taux_suivant|taux_moyenne_mobile_3|
+-----------------------+-------------------+---------------+--------------+------------+---------------------+
|18 juin 1940 - Buzenval|2020-11-26 12:59:00|0.12           |NULL          |0.12        |0.12                 |
|18 juin 1940 - Buzenval|2020-11-26 13:06:00|0.12           |0.12          |0.12        |0.12                 |
|18 juin 1940 - Buzenval|2020-11-26 13:21:00|0.12           |0.12          |0.12        |0.12                 |
|18 juin 1940 - Buzenval|2020-11-26 13:32:00|0.12           |0.12          |0.12        |0.12                 |
|18 juin 1940 - Buzenval|2020-11-26 13:47:00|0.12           |0.12          |0.12        |0.12                 |
|18 juin 1940 - Buzenval|2020-11-26 14:25:00|0.12           |0.12          |0.16        |0.1333333333333

In [14]:
# Classement des stations par taux d'occupation moyen, a chaque heure de la journee
# ROW_NUMBER() numerote les lignes dans chaque partition (ici : chaque heure)

df_moyennes_par_heure = (
    df.groupBy("heure", "nom_station")
      .agg(F.avg("taux_occupation").alias("taux_occupation_moyen"))
)

fenetre_heure = Window.partitionBy("heure").orderBy(F.desc("taux_occupation_moyen"))

df_rank = (
    df_moyennes_par_heure
    .withColumn("rang", F.row_number().over(fenetre_heure))
    .filter(F.col("rang") <= 2)
    .orderBy("heure", "rang")
)
df_rank.show(48, truncate=False)


+-----+------------------------------------+---------------------+----+
|heure|nom_station                         |taux_occupation_moyen|rang|
+-----+------------------------------------+---------------------+----+
|0    |Grenelle - Dr Finlay                |0.8970676923076925   |1   |
|0    |Place Charles Michels               |0.8623630769230768   |2   |
|1    |Route de Sèvres - Porte de Bagatelle|0.8832199999999998   |1   |
|1    |Grenelle - Dr Finlay                |0.8811146666666665   |2   |
|2    |Grenelle - Dr Finlay                |0.8788273972602745   |1   |
|2    |Route de Sèvres - Porte de Bagatelle|0.8724424657534247   |2   |
|3    |Route de Sèvres - Porte de Bagatelle|0.8746754716981131   |1   |
|3    |Grenelle - Dr Finlay                |0.8558660377358496   |2   |
|4    |Route de Sèvres - Porte de Bagatelle|0.8714636363636366   |1   |
|4    |Grenelle - Dr Finlay                |0.8528440559440562   |2   |
|5    |Route de Sèvres - Porte de Bagatelle|0.8781104395604402  

In [15]:
# Calcul de la variation du taux sur une heure glissante
# UNBOUNDED PRECEDING -> ligne actuelle = cumul depuis le debut de la partition

fenetre_cumul = (
    Window
    .partitionBy("nom_station")
    .orderBy("horodatage")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

# Variation par rapport au snapshot precedent (delta instantane)
station_cible = "Château - République"   # pas d'identifiant numerique dans nos donnees -- on utilise le nom

df_delta = (
    df
    .filter(F.col("nom_station") == station_cible)
    .withColumn("taux_precedent", F.lag("taux_occupation").over(Window.partitionBy("nom_station").orderBy("horodatage")))
    .withColumn("delta_instantane", F.round(F.col("taux_occupation") - F.col("taux_precedent"), 4))
    .withColumn("taux_cumule_moyen", F.round(F.avg("taux_occupation").over(fenetre_cumul), 4))
    .select("nom_station", "horodatage", "taux_occupation", "taux_precedent", "delta_instantane", "taux_cumule_moyen")
)
df_delta.show(truncate=False)


+--------------------+-------------------+---------------+--------------+----------------+-----------------+
|nom_station         |horodatage         |taux_occupation|taux_precedent|delta_instantane|taux_cumule_moyen|
+--------------------+-------------------+---------------+--------------+----------------+-----------------+
|Château - République|2020-11-26 12:59:00|0.2692         |NULL          |NULL            |0.2692           |
|Château - République|2020-11-26 12:59:00|0.15           |0.2692        |-0.1192         |0.2096           |
|Château - République|2020-11-26 13:06:00|0.1923         |0.15          |0.0423          |0.2038           |
|Château - République|2020-11-26 13:06:00|0.15           |0.1923        |-0.0423         |0.1904           |
|Château - République|2020-11-26 13:21:00|0.1923         |0.15          |0.0423          |0.1908           |
|Château - République|2020-11-26 13:21:00|0.25           |0.1923        |0.0577          |0.2006           |
|Château - Républiq


> **⚠️ A savoir avant de lancer cette section (Delta Lake)**
>
> Il faut installer le package correspondant a ta version de PySpark (`pyspark
> 4.2.0` d'apres ton venv) :
> ```
> pip install delta-spark==4.2.0
> ```
> Au premier lancement de la cellule suivante, Spark telecharge automatiquement
> le jar Delta Lake depuis Maven Central (`repo1.maven.org`) -- il faut une
> connexion internet la premiere fois, ensuite le jar reste en cache local
> (`~/.ivy2`). Je n'ai pas pu executer/valider cette section moi-meme : le
> reseau du bac a sable cloud bloque ce depot Maven, et le shell sur ta
> machine etait indisponible pendant cette session. Le code ci-dessous est
> ecrit et relu avec soin, mais lance-le et dis-moi si quelque chose casse.
>
> Egalement adapte a la vraie periode de tes donnees : le notebook d'origine
> travaillait sur "2022" puis "2023" -- ici, c'est "2020" (nov-dec) puis
> "2021" (jan-fev), la vraie periode couverte par `historique_stations.csv`.


from delta.tables import DeltaTable

# Ecriture initiale : toutes les donnees de 2020 (nov-dec -- voir la note
# ci-dessus sur la periode reelle des donnees)
df_2020 = df.filter(F.col("annee") == 2020)

t0 = time.perf_counter()
(
    df_2020
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("annee", "mois")
    .save(str(DELTA_DISPONIBLE))
)
print(f"Ecriture 2020 : {time.perf_counter()-t0:.1f} s  --  {df_2020.count():,} lignes")

# Verification de la structure Delta
fichiers_delta = list(Path(DELTA_DISPONIBLE).rglob("*.parquet"))
log_delta      = list(Path(DELTA_DISPONIBLE / "_delta_log").glob("*.json"))
print(f"Fichiers Parquet : {len(fichiers_delta)}")
print(f"Entrees dans le transaction log : {len(log_delta)}")


In [ ]:
# Ajout des donnees 2021 (jan-fev) -- mode "append"
df_2021 = df.filter(F.col("annee") == 2021)

t0 = time.perf_counter()
(
    df_2021
    .write
    .format("delta")
    .mode("append")
    .partitionBy("annee", "mois")
    .save(str(DELTA_DISPONIBLE))
)
print(f"Ajout 2021 : {time.perf_counter()-t0:.1f} s  --  {df_2021.count():,} lignes")

# Historique des versions : chaque operation cree une nouvelle version
delta_table = DeltaTable.forPath(spark, str(DELTA_DISPONIBLE))
delta_table.history().select(
    "version", "timestamp", "operation", "operationParameters"
).show(truncate=False)


In [ ]:
# Ajout des données 2023 -- mode "append"
df_2023 = df.filter(F.col("annee") == 2023)

t0 = time.perf_counter()
(
    df_2023 # Enregistrer les données au format DeltaLake, en mode ajout
)
print(f"Ajout 2023 : {time.perf_counter()-t0:.1f} s  --  {df_2023.count():,} lignes")

# Historique des versions : chaque opération crée une nouvelle version
delta_table = DeltaTable.forPath(spark, str(DELTA_DISPONIBLE))
delta_table.history().select(
    "version", "timestamp", "operation", "operationParameters"
).show(truncate=False)


# Lecture de la version 0 (uniquement les donnees 2020)
df_v0 = spark.read.format("delta").option("versionAsOf", 0).load(str(DELTA_DISPONIBLE))
print(f"Version 0 (2020 uniquement) : {df_v0.count():,} lignes")

# Lecture de la version courante
df_current = spark.read.format("delta").load(str(DELTA_DISPONIBLE))
print(f"Version courante (2020+2021) : {df_current.count():,} lignes")

# Enregistrement comme vue SQL pour la suite
df_current.createOrReplaceTempView("disponibilite_delta")


In [ ]:
# Lecture de la version 0 (uniquement les données 2022)
df_v0 = (
# TODO Lecture des données
)
print(f"Version 0 (2022 uniquement) : {df_v0.count():,} lignes")

# Lecture de la version courante
df_current = # TODO
print(f"Version courante (2022+2023) : {df_current.count():,} lignes")

# Enregistrement comme vue SQL pour la suite
df_current.createOrReplaceTempView("disponibilite_delta")


### `MERGE INTO` : mise à jour incrémentale

`MERGE INTO` est l'opération la plus puissante de Delta Lake. Elle permet de
**synchroniser** une table cible avec une table source en une seule passe :
insertions des nouvelles lignes, mises à jour des lignes existantes,
suppressions optionnelles.

Cas d'usage typique : arrivée quotidienne d'un nouveau batch de snapshots.


In [ ]:
# MERGE INTO : upsert (update + insert)
# (pas d'identifiant numerique de station dans nos donnees -- on matche sur
# nom_station + horodatage, voir Session 1)
(
    delta_table.alias("cible")
    .merge(
        df_batch.alias("source"),
        "cible.nom_station = source.nom_station AND cible.horodatage = source.horodatage"
    )
    .whenMatchedUpdateAll()     # si correspondance : on ecrase toutes les colonnes
    .whenNotMatchedInsertAll()  # si pas de correspondance : on insere
    .execute()
)

# Verification : la table a une nouvelle version
delta_table.history().select(
    "version", "timestamp", "operation",
    "operationMetrics"
).show(5, truncate=False)


In [ ]:
# [EXERCICE]
# Le MERGE precedent a mis a jour des lignes existantes.
# Utilisez le time-travel pour comparer le taux_occupation moyen
# de janvier (2021 -- le seul janvier present dans nos donnees reelles)
# AVANT et APRES le merge.
#
# Indice : lisez la version 1 (avant merge, juste apres l'ajout 2021) et la
# version courante, puis comparez avec une agregation.
# ──────────────────────────────────────────────────────────────────────────

df_v1 = spark.read.format("delta").option("versionAsOf", 1).load(str(DELTA_DISPONIBLE))
df_courante = spark.read.format("delta").load(str(DELTA_DISPONIBLE))

taux_avant = (
    df_v1.filter((F.col("annee") == 2021) & (F.col("mois") == 1))
         .agg(F.round(F.avg("taux_occupation"), 4).alias("taux_moyen"))
         .collect()[0]["taux_moyen"]
)
taux_apres = (
    df_courante.filter((F.col("annee") == 2021) & (F.col("mois") == 1))
         .agg(F.round(F.avg("taux_occupation"), 4).alias("taux_moyen"))
         .collect()[0]["taux_moyen"]
)

print(f"Taux moyen janvier 2021 AVANT le merge : {taux_avant}")
print(f"Taux moyen janvier 2021 APRES le merge : {taux_apres}")
print(f"Difference : {round(taux_apres - taux_avant, 4)}")


In [ ]:
# [EXERCICE]
# Le MERGE précédent a mis à jour des lignes existantes.
# Utilisez le time-travel pour comparer le taux_occupation moyen
# de janvier 2022 AVANT et APRÈS le merge.
#
# Indice : lisez la version 1 (avant merge) et la version courante,
# puis comparez avec une agrégation.
# ──────────────────────────────────────────────────────────────────────────

# Votre code ici :
